# Урок 8.3. POST /api/search: двухступенчатый поиск

Первый боевой эндпоинт: семантический поиск по документации. Внутри — весь
модуль 2 (эмбеддинги, Qdrant) и лучшее из модуля 4 (двухступенчатая схема
с reranker), упакованные в честный HTTP-контракт.

🎯 **Цели урока:**

- Проследить путь запроса: роутер → RagService → эмбеддинг → ANN → reranker.
- Понять, почему CPU-bound код в async-приложении уезжает в thread pool.
- Живьём сравнить поиск с reranker'ом и без — качество, шкалы, цена.
- Разобрать контракт: две несравнимые шкалы оценок в одном ответе.

**Что понадобится:** Qdrant с коллекцией из урока 8.2, Ollama (для /ready),
модели e5 и cross-encoder в кэше HF.

## 1. Путь запроса

```
POST /api/search {"query": "как откатить деплой?", "top_k": 3}
  │ routers/search.py: валидация SearchRequest (Pydantic, 422 на мусор)
  ▼
RagService.search()                        ← services/rag.py
  │ 1. embed_query_async("как откатить...")     → вектор 384
  │ 2. vectorstore.search(vector, limit=fetch_k) → 20 кандидатов (ANN, HNSW)
  │ 3. фильтр по search_min_score
  │ 4. reranker.rerank_async(query, кандидаты)   → точная пересортировка
  ▼
top_k результатов → SearchResponse (score + rerank_score + источник)
```

Ключевое решение — **fetch_k ≠ top_k**: у Qdrant просим 20 кандидатов (дёшево,
ANN), reranker читает пары «запрос+чанк» целиком (дорого, зато точно) и
возвращает лучшие 5. Это та самая двухступенчатая схема из уроков 2.6 и 4.4,
теперь в проде. Выключен reranker — просто отдаём top_k первых от Qdrant.

## 2. Async-ловушка: CPU-bound в event loop

`SentenceTransformer.encode` — тяжёлая CPU-работа. Вызвать её прямо
в async-обработчике — значит **заморозить весь event loop**: пока считается
вектор, сервис не отвечает даже на `/health` (мы измеряли эту лесенку в уроке
3.2 на блокирующем эндпоинте). Поэтому в `EmbeddingService`:

```python
async def embed_query_async(self, text: str) -> list[float]:
    return await asyncio.to_thread(self.embed_query, text)   # в thread pool
```

То же — в reranker'е. Правило: **async-приложению позволено ждать только IO;
всё, что жуёт CPU, уезжает в поток или процесс.**

Поднимем сервис и посмотрим на всё это вживую.

In [ ]:
import json
import subprocess
import sys
import time
from pathlib import Path

import httpx

BASE = "http://127.0.0.1:8010"

server = subprocess.Popen(
    [sys.executable, "-X", "utf8", "-m", "uvicorn", "app.main:app",
     "--host", "127.0.0.1", "--port", "8010"],
    cwd="project", stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
print("uvicorn pid:", server.pid)

# Старт грузит e5 (и cross-encoder, если включён) - терпеливо ждём /health.
with httpx.Client(timeout=10) as probe:
    for attempt in range(240):
        try:
            if probe.get(f"{BASE}/health").status_code == 200:
                print(f"поднялся за ~{attempt * 0.5:.0f} c")
                break
        except httpx.HTTPError:
            pass
        if server.poll() is not None:
            raise RuntimeError("uvicorn умер при старте - смотрите логи запуска")
        time.sleep(0.5)
    else:
        raise RuntimeError("сервис не поднялся за 2 минуты")

    ready = probe.get(f"{BASE}/ready")
    print("/ready:", ready.status_code, json.dumps(ready.json(), ensure_ascii=False))

client = httpx.Client(timeout=300)   # общий клиент для ячеек урока

In [ ]:
QUESTIONS = [
    "как откатить деплой на проде?",
    "что делать при коммите секрета в git?",
    "за сколько минут дежурный должен отреагировать ночью?",
]

for question in QUESTIONS:
    response = client.post(f"{BASE}/api/search", json={"query": question, "top_k": 3})
    body = response.json()
    print(f"\n== {question}  ({body['took_ms']} мс)")
    for res in body["results"]:
        print(f"   cos={res['score']:.3f}  rerank={res['rerank_score']:+.2f}  "
              f"{res['source']}#{res['chunk_index']}  «{res['text'][:60]}...»")

Три запроса — три поучительные картинки (мой прогон):

- **«Коммит секрета»**: правильный чанк первый по обеим шкалам, и посмотрите
  на rerank-зазор — **+8.39 против +2.72** у второго места. Уверенный отрыв —
  почерк cross-encoder'а, который мы впервые увидели в модуле 2: когда ответ
  действительно есть, CE не сомневается.
- **«Дежурный ночью»**: чанк «## Дежурство» с «15 минут в рабочее время
  и 30 ночью» — первый, +5.47. Обратите внимание, что второй кандидат
  («## CI») по косинусу почти неотличим от первого (0.840 против 0.879) —
  а по rerank-шкале между ними пропасть.
- **«Откат деплоя»**: top-1 верный («## Откат»), но все rerank-логиты
  в минусе. CE честен: лексически вопрос и «повторный запуск job с предыдущим
  тегом» связаны слабо, уверенности нет. Мораль: пороги по rerank-шкале тоже
  нельзя назначать «на глазок» — только калибровкой (урок 3.4).

И общее: все косинусы живут в полосе 0.82–0.93 — плотная шкала e5, по которой
можно сортировать, но нельзя судить об «уверенности». Разделяет кандидатов
именно reranker.

In [ ]:
# Тот же запрос с reranker и без: сравниваем порядок и цену
tricky = "как откатить деплой на проде?"
for use_reranker in (True, False):
    # прогревочный прогон + замер (первый запрос платит за прогрев моделей)
    client.post(f"{BASE}/api/search", json={"query": tricky, "use_reranker": use_reranker})
    response = client.post(f"{BASE}/api/search",
                           json={"query": tricky, "top_k": 3, "use_reranker": use_reranker})
    body = response.json()
    label = "с reranker   " if use_reranker else "без reranker "
    order = [f"{r['source'].split('-')[0]}#{r['chunk_index']}" for r in body["results"]]
    print(f"{label} {body['took_ms']:>4} мс  порядок: {order}")

Два наблюдения из живого сравнения:

- **Цена**: ~590 мс с reranker против ~40 без — примерно **в 14 раз дороже**
  (CPU cross-encoder на 20 кандидатах, тёплый сервер). Для внутреннего
  ассистента — приемлемо; для высоконагруженного API — повод оставить
  `use_reranker` per-запросным флагом, что мы и сделали.
- **Качество**: без reranker весь топ-3 — соседние секции одного
  `05-deploy-guide.md` (по косинусу свои секции всегда рядом). Reranker
  поставил вторым чанк «Обратная совместимость» из `09-db-migrations.md` —
  а это ровно то, что обязан знать человек, откатывающий прод: **миграции при
  откате не откатываются**. Вторая ступень не только переставляет — она
  разбавляет монополию одного документа семантически связанными фактами
  из других.

`took_ms` из ответа — это то самое число, которое в проде превращается
в гистограмму `rag_search_duration_seconds` (урок 8.5): сегодняшние 590 мс —
завтрашний baseline для алертов.

In [ ]:
# Контракт на границе: фильтр по источнику и защита от мусора
response = client.post(f"{BASE}/api/search",
                       json={"query": "правила аппрувов", "top_k": 2,
                             "source": "03-git-flow.md"})
for res in response.json()["results"]:
    print(f"{res['source']}#{res['chunk_index']}: {res['text'][:70]}...")

print("\nпустой query ->", client.post(f"{BASE}/api/search", json={"query": ""}).status_code)
print("top_k=100    ->", client.post(f"{BASE}/api/search",
                                     json={"query": "x", "top_k": 100}).status_code)

Фильтр `source` — это payload-фильтр Qdrant (filtered HNSW из урока 2.2:
фильтруем **во время** обхода графа, не после), а 422 на мусор — работа Pydantic
ещё до нашего кода. Контракт держит границу с обеих сторон.

## ⚠️ Частые ошибки

1. **`encode` прямо в async-обработчике.** Сервис «замирает» на каждый запрос —
   и «почему-то» падают health-чеки под нагрузкой. `asyncio.to_thread`.
2. **Сравнение score и rerank_score.** Косинус ~0.8 и логит −0.6 — разные
   вселенные. В контракте это два отдельных поля с явным предупреждением
   в описании; сортировка — по rerank_score, когда он есть.
3. **`fetch_k == top_k` при включённом reranker.** Пересортировка пяти
   кандидатов почти ничего не даёт — reranker'у нужна широкая сеть (20+),
   из которой он вытащит зарытое.
4. **Фильтр min_score «на глазок».** У e5 все осмысленные косинусы живут
   в узкой полосе ~0.75–0.87 — порог 0.5 не отфильтрует ничего, порог 0.85
   убьёт половину хороших ответов. Пороги калибруются на своих данных
   (урок 3.4), поэтому дефолт — 0.0.

## Мини-квиз

**1. Почему reranker пересортировывает 20 кандидатов, а не всю коллекцию?**

<details><summary>Ответ</summary>

Cross-encoder читает пару «запрос+чанк» полным проходом модели — O(N) прогонов.
На 61 чанке можно и всю базу, но на 100 тысячах — часы. ANN за миллисекунды
отбирает разумных кандидатов, дорогая точность тратится только на них
(урок 2.6: «cross-encoder по базе — часы, по 30 кандидатам — секунды»).
</details>

**2. Что сломается, если убрать `asyncio.to_thread` вокруг encode?**

<details><summary>Ответ</summary>

Каждый `encode` (десятки миллисекунд CPU) блокирует event loop целиком:
параллельные запросы, `/health`, стримы — всё ждёт. Под нагрузкой сервис
выглядит «зависшим», хотя CPU занят работой. Классика из урока 3.2.
</details>

**3. Клиент жалуется: «score почти одинаковый у всех результатов, как выбирать?» Что ответить?**

<details><summary>Ответ</summary>

Так устроены плотные эмбеддинги e5: рабочая полоса косинусов узкая, и это
нормально — score годится для сортировки, а не как «вероятность правильности».
Для выбора включите reranker: его логиты растянуты и лучше разделяют
кандидатов (и это ровно то, что видно в нашем живом сравнении).
</details>

## 🎓 Итоги

- Путь: Pydantic-контракт → RagService → эмбеддинг (в thread pool!) → ANN
  по fetch_k → reranker → top_k.
- Двухступенчатая схема в проде: широкая дешёвая сеть + дорогая точная
  пересортировка, выключаемая конфигом и параметром запроса.
- Две шкалы в ответе — осознанный контракт: cosine для отбора, rerank-логит
  для порядка; сравнивать их между собой нельзя.
- Payload-фильтр source и валидация 422 держат границу API.

**Дальше — урок 8.4:** `/api/generate` — grounding-промпт, цитирование [N],
SSE-стриминг с источниками до первого токена и история диалогов.

In [ ]:
client.close()
server.terminate()
server.wait(timeout=15)
print("uvicorn остановлен")